In [12]:
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import scipy.stats as stats
import scipy.io as sio
from sklearn.metrics import roc_auc_score as auROC
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
np.set_printoptions(suppress=True)

In [13]:
alphalevel = 0.05
bh_correction = 'no'
extractedsignalsonly = 'no'
mineventstoanalyze = 3
save_aligned_data = 'yes' # FIXME: changed to yes
save_auc_array = 'no'
delete_overlaps = 'yes'
separation_requirement = 1000
frameaveraging = 4
framerate = 30
timebetweenframes = 33.333333
averagedframerate = framerate/frameaveraging
z_score_data = 'yes'
pre_window_size = int(10*averagedframerate)
window_size =  int((pre_window_size*2)+(1.6*averagedframerate))
post_window_size = window_size - pre_window_size
baselinefirstframe = 0
baselinelastframe = int(3*averagedframerate)
infusionframe = int(pre_window_size+(1.6*averagedframerate))
aucfirstframe = int(pre_window_size-(5*averagedframerate))
auclastframe = int(pre_window_size+(5*averagedframerate))
eventofinterest = 'activeleverall'
basedir = r'../data'

In [14]:
def analyze_single_session(indir, window_size, pre_window_size):
    tempfiles = next(os.walk(indir))[2]
    npyfiles = [f for f in tempfiles if os.path.splitext(f)[1]=='.npy' and 'extractedsignals_raw' in f]
    matfiles = [f for f in tempfiles if os.path.splitext(f)[1]=='.mat']
    if len(npyfiles) > 1:
        npyfile = [f for f in tempfiles if os.path.splitext(f)[1]=='.npy' and 'extractedsignals_raw' in f and not 'part2' in f and not 'part3' in f and not 'part4' in f]
        npyfile = npyfile [0]
        matfile = [f for f in tempfiles if os.path.splitext(f)[1]=='.mat' and not 'results' in f and not 'part2' in f and not 'part3' in f and not 'part4' in f]
        matfile = matfile [0]
    else:
        npyfile = npyfiles[0]
        matfile = matfiles[0]
    signals = np.squeeze(np.load(os.path.join(indir, npyfile)))
    numrois = signals.shape[0]
    behaviordata = sio.loadmat(os.path.join(indir, matfile))
    eventlog = np.squeeze(behaviordata['eventlog'])
    licks = np.squeeze(behaviordata['licks'])
    lastframe_timestamp_part1 = np.max(eventlog)
    if len(npyfiles) > 1:
        npyfiles2 = [f for f in tempfiles if os.path.splitext(f)[1]=='.npy' and 'extractedsignals_raw'and 'part2' in f]
        matfiles2 = [f for f in tempfiles if os.path.splitext(f)[1]=='.mat' and 'part2' in f and not 'results' in f]
        npyfile2 = npyfiles2[0]
        matfile2 = matfiles2[0]
        signals2 = np.squeeze(np.load(os.path.join(indir, npyfile2))) 
        signals = np.hstack((signals, signals2))
        behaviordata2 = sio.loadmat(os.path.join(indir, matfile2))
        eventlog2 = np.squeeze(behaviordata2['eventlog'])
        eventlog2[:,1] = eventlog2[:,1]+lastframe_timestamp_part1  #adding the last frame to the second column of data in eventlog
        eventlog = np.concatenate((eventlog,eventlog2))
        lastframe_timestamp_part2 = np.max(eventlog)
        licks2 = np.squeeze(behaviordata2['licks'])
        licks2 = licks2+lastframe_timestamp_part1
        licks = np.concatenate ([licks, licks2])
    if len(npyfiles) > 2:
        npyfiles3 = [f for f in tempfiles if os.path.splitext(f)[1]=='.npy' and 'extractedsignals_raw'and 'part3' in f]
        matfiles3 = [f for f in tempfiles if os.path.splitext(f)[1]=='.mat' and 'part3' in f and not 'results' in f]
        npyfile3 = npyfiles3[0]
        matfile3 = matfiles3[0]
        signals3 = np.squeeze(np.load(os.path.join(indir, npyfile3))) 
        signals = np.hstack((signals, signals3))
        behaviordata3 = sio.loadmat(os.path.join(indir, matfile3))
        eventlog3 = np.squeeze(behaviordata3['eventlog'])
        eventlog3[:,1] = eventlog3[:,1]+lastframe_timestamp_part2  #adding the last frame to the second column of data in eventlog
        eventlog = np.concatenate((eventlog,eventlog3))
        lastframe_timestamp_part3 = np.max(eventlog)
        licks3 = np.squeeze(behaviordata3['licks'])
        licks3 = licks3+lastframe_timestamp_part2
        licks = np.concatenate ([licks, licks3])
    if len(npyfiles) > 3:
        npyfiles4 = [f for f in tempfiles if os.path.splitext(f)[1]=='.npy' and 'extractedsignals_raw' and 'part4' in f]
        matfiles4 = [f for f in tempfiles if os.path.splitext(f)[1]=='.mat' and 'part4' in f and not 'results' in f]
        npyfile4 = npyfiles4[0]
        matfile4 = matfiles4[0]
        signals4 = np.squeeze(np.load(os.path.join(indir, npyfile4))) 
        signals = np.hstack((signals, signals4))
        behaviordata4 = sio.loadmat(os.path.join(indir, matfile4))
        eventlog4 = np.squeeze(behaviordata4['eventlog'])
        eventlog4[:,1] = eventlog4[:,1]+lastframe_timestamp_part3  #adding the last frame to the second column of data in eventlog
        eventlog = np.concatenate((eventlog,eventlog4))
        lastframe_timestamp_part4 = np.max(eventlog)
        licks4 = np.squeeze(behaviordata4['licks'])
        licks4 = licks4+lastframe_timestamp_part3
        licks = np.concatenate ([licks, licks4])
    if extractedsignalsonly == 'yes':
        return signals
    activelever = eventlog[eventlog[:,0]==22,1]
    activelevertimeout = eventlog[eventlog[:,0]==222,1]
    inactivelever = eventlog[eventlog[:,0]==21,1]
    inactivelevertimeout = eventlog[eventlog[:,0]==212,1]
    cues = eventlog[eventlog[:,0]==7,1]
    infusions = eventlog[eventlog[:,0]==4,1]
    if isinstance(window_size, str): 
        return activelever, activelevertimeout
    if delete_overlaps == 'yes':
        if 'active' in eventofinterest:
            newarray = np.array([])
            temp = np.sort(np.hstack((activelever, activelevertimeout)))
            temp = np.delete(temp, np.argwhere(np.ediff1d(temp)<separation_requirement)+1)
            if eventofinterest == 'activelever':
                for i in range(len(temp)):
                    if temp[i] in activelever:
                        newarray = np.append(newarray, temp[i])
                activelever = newarray    
            if eventofinterest == 'activelevertimeout':
                for i in range(len(temp)):
                    if temp[i] in activelevertimeout:
                        newarray = np.append(newarray, temp[i])
                activelevertimeout = newarray
            if eventofinterest == 'activeleverall':
                activeleverall = temp
        elif 'inactive' in eventofinterest:
            newarray = np.array([])
            temp = np.sort(np.hstack((inactivelever, inactivelevertimeout)))
            temp = np.delete(temp, np.argwhere(np.ediff1d(temp)<separation_requirement)+1)
            if eventofinterest == 'inactivelever':
                for i in range(len(temp)):
                    if temp[i] in inactivelever:
                        newarray = np.append(newarray, temp[i])
                inactivelever = newarray    
            if eventofinterest == 'inactivelevertimeout':
                for i in range(len(temp)):
                    if temp[i] in inactivelevertimeout:
                        newarray = np.append(newarray, temp[i])
                inactivelevertimeout = newarray
            if eventofinterest == 'inactiveleverall':
                inactiveleverall = temp 
    else:
        activeleverall = np.sort(np.hstack((activelever, activelevertimeout))) ### np.sort integrates them in array rather than keeping separated
        inactiveleverall = np.sort(np.hstack((inactivelever, inactivelevertimeout))) ### np.sort integrates them in array rather than keeping separated
    if eventofinterest == 'activelever':
        events = activelever
    elif eventofinterest == 'activelevertimeout':
        events = activelevertimeout
    elif eventofinterest == 'inactivelever':
        events = inactivelever
    elif eventofinterest == 'inactivelevertimeout':
        events = inactivelevertimeout
    elif eventofinterest == 'cues':
        events = cues
    elif eventofinterest == 'infusions':
        events = infusions
    elif eventofinterest == 'activeleverall':
        events = activeleverall
    elif eventofinterest == 'inactiveleverall':
        events = inactiveleverall
    if len(events) < 2:
        return np.nan*np.ones((2, window_size)), np.nan*np.ones((2, window_size)), np.nan*np.ones((2, window_size)), np.nan*np.ones((2, window_size))
    if animal == 'CTL1' or animal == 'ER-L1' or animal == 'ER-L2' or animal == 'IG-19' or animal == 'IG-28' or animal == 'PGa-T1' or animal == 'XYZ':
        frame_timestamps = np.array(assumed_frame_timestamps) ###Fixes issue for finding behavior IF YOU DON"T HAVE FRAME INFO
    else:
        frame_timestamps = fix_any_dropped_frames(eventlog[eventlog[:,0]==9,1]) ### Fixes dropped frames for data that has timestamps, using function below
    frame_timestamps = frame_timestamps[::frameaveraging] ###incorporates averaging into timestamp array
    if signals.shape[1] > frame_timestamps.shape[0]:
        signals = signals[:,:frame_timestamps.shape[0]-1] ###cuts signals so it's not longer than the frame timestamps
    signals /= np.nanmean(signals, axis=1)[:, None] ###averages signals around 1, rather than pixel intensity
    if z_score_data == 'yes':
        for neuron in range(signals.shape[0]):
            mean = np.nanmean(signals[neuron])
            std = np.nanstd(signals[neuron])
            signals[neuron] = (signals[neuron] - mean)/std
    signalsT = signals.T
    if len(events) < mineventstoanalyze:
        return np.nan*np.ones((2, window_size)), np.nan*np.ones((2, window_size)), np.nan*np.ones((2, window_size)), np.nan*np.ones((2, window_size))

    def calculate_aligneddata_forevent(signalsT2, events2):
        framenumberfor_eventofinterest = np.squeeze(framenumberforevent(events2, frame_timestamps)) ### Uses "framenumberforevent" function below
        numtrials = framenumberfor_eventofinterest.shape[0]
        alignedevents = np.NAN*np.zeros([numtrials,window_size,numrois])
        for i in np.flip(range(numtrials)):
            eventindex = framenumberfor_eventofinterest[i]
            if np.isfinite(eventindex) and eventindex > pre_window_size and eventindex < np.shape(signalsT2)[0]-post_window_size:
                eventindex = int(eventindex)
                alignedevents[i, :, :] = signalsT2[eventindex-pre_window_size:eventindex+post_window_size, :]
            else:
                alignedevents = np.delete(alignedevents, i, axis = 0)
        return alignedevents

    alignedevents = calculate_aligneddata_forevent(signalsT, events)
    for i in range(signals.shape[0]):
        if np.isnan(np.mean(signals[i,:])):
            print(animal, fov, 'IMAGE J ROI.ZIP CELL NUMBER %s HAS NaNs AND SHOULD BE CHANGED'%(i+1))
    alignedevents = np.swapaxes(alignedevents, 0,2)
    popevents = np.nanmean(alignedevents, axis=2)
    return popevents, alignedevents, events, events


### THIS FUNCTION IS ALSO USED BY ANALYZE SINGLE SESSION ###
def framenumberforevent(event, frame_timestamps):
    framenumber = np.nan*np.zeros(event.shape)
    for ie, e in enumerate(event):
        if np.isnan(e):
            framenumber[ie] = np.nan
        else:
            temp = np.nonzero(frame_timestamps<=e)[0]
            if temp.shape[0]>0:
                framenumber[ie] = np.nonzero(frame_timestamps<=e)[0][-1]
            else:
                framenumber[ie] = 0
    return framenumber


In [15]:
if save_aligned_data == 'yes':
    def fix_any_dropped_frames(frame_timestamps):
        first_frame = np.array([0])
        last_frame = np.array([int(np.max(frame_timestamps)+(500*timebetweenframes))])
        frame_index_temp = np.concatenate((first_frame,frame_timestamps, last_frame))
        frames_missed = []
        for i in range(len(frame_index_temp)-1):
                numframes_missed = int(np.round((frame_index_temp[i+1]-frame_index_temp[i])\
                    /timebetweenframes)-1)
                if numframes_missed > 0: 
                    for j in range(numframes_missed):
                        frame_missed = np.array([frame_index_temp[i] + (int(timebetweenframes * (j+1)))])
                        frames_missed = np.concatenate((frames_missed, frame_missed))
        corrected_frame_index = np.array(sorted(np.concatenate((frame_index_temp, frames_missed))))
        return corrected_frame_index

    def fix_assumed_frames(frames):
        dropped_frames = []
        diff_frames = np.diff(frames)
        inter_frame_interval = 33
        frame_drop_idx = np.where(diff_frames>1.5*inter_frame_interval)[0]
        for idx in frame_drop_idx:
            numframesdropped = int(np.round((frames[idx+1]-frames[idx])/(inter_frame_interval+0.0))-1)
            temp = [frames[idx]+a*inter_frame_interval for a in range(1,numframesdropped+1)]
            dropped_frames.extend(temp)
        corrected_frames = np.sort(np.concatenate((frames, np.array(dropped_frames))))
        return corrected_frames

    behaviordata_noframes = sio.loadmat(r'/home/otis-lab/Desktop/pynapse/data/empty.mat')
    eventlog_noframes = np.squeeze(behaviordata_noframes['eventlog'])
    max_of_eventlog_noframes = max(eventlog_noframes[:,1])
    length_of_eventlog_noframes = len(eventlog_noframes[:,1])
    x = np.vstack((eventlog_noframes, eventlog_noframes, eventlog_noframes))
    x[length_of_eventlog_noframes:,1]= x[length_of_eventlog_noframes:,1]+max_of_eventlog_noframes
    x[length_of_eventlog_noframes*2:,1]= x[length_of_eventlog_noframes*2:,1]+(2*max_of_eventlog_noframes)
    eventlog_noframes = x
    assumed_frames = fix_any_dropped_frames(eventlog_noframes[eventlog_noframes[:,0]==9,1])
    assumed_frame_timestamps = fix_assumed_frames(eventlog_noframes[eventlog_noframes[:,0]==9,1])


In [16]:
def calculate_auROC(x,y,offset_to_zero=True):
    U, p = stats.mannwhitneyu(x,y)
    labels = np.concatenate((np.ones(x.shape), np.zeros(y.shape)))
    data = np.concatenate((x,y))
    A = auROC(labels, data)
    if offset_to_zero:
        return (2*(A-0.5), p)
    else:
        return (A, p)
    
def Benjamini_Hochberg_correction(vector_of_pvals, alpha = 0.05):

    sortedpvals = np.sort(vector_of_pvals)
    orderofpvals = np.argsort(vector_of_pvals)
    m = sortedpvals[np.isfinite(sortedpvals)].shape[0] #Total number of hypotheses
    for i in range(m):
        if sortedpvals[i] > (i+1)*alpha/m:
            k = i
            break
        elif i == m-1:
            k = m-1
    correctedpvals = np.copy(vector_of_pvals)
    correctedpvals[orderofpvals[k:]] = 1
    correctedpvals[np.isnan(vector_of_pvals)] = np.nan
    return correctedpvals

def iterate_dirs(basedir, days):
    for day in sorted(days):
        animals = next(os.walk(os.path.join(basedir, day)))[1]
        for animal in sorted(animals):
            FOVs = next(os.walk(os.path.join(basedir, day, animal)))[1]
            for fov in sorted(FOVs):
                yield basedir, day, animal, fov

In [17]:
tempdays = next(os.walk(os.path.join(basedir)))[1]
days = []
for t in tempdays:
    if t!= 'Cascade' and t!= 'CellTracking' and t!= 'Codes' and t != '.ipynb_checkpoints' and t != 'Other' and t!= 'Results':
        days = np.append(days, t)
numdays = len(days)
popevents_day = {}
popevents_fov = {}
for basedir, day, animal, fov in iterate_dirs(basedir, sorted(days)):
    if day not in popevents_fov:
        popevents_day[day] = np.nan*np.ones((1, window_size))
        popevents_fov[day] = {}
        print(day)   
    if save_aligned_data == 'yes':
        temp1, temp2, temp3, temp4 = analyze_single_session(os.path.join(basedir, day, animal, fov), window_size, pre_window_size)
    tempfiles = next(os.walk(os.path.join(basedir, day, animal, fov)))[2]
    if 'popevents_%s_%s.npy'%(eventofinterest, separation_requirement) in tempfiles: 
        if animal not in popevents_fov[day]:
            popevents_fov[day][animal] = {}
        temp1 = np.load(os.path.join(basedir, day, animal, fov, 'popevents_%s_%s.npy'%(eventofinterest, separation_requirement)))  
        if np.isfinite(np.nanmean(temp1)):
            popevents_day[day] = np.vstack((popevents_day[day], temp1))
            popevents_fov[day][animal][fov] = temp1
            if day == '4 LastExt':
                print(day, animal, fov, temp1.shape)
            baseline = np.mean(popevents_fov[day][animal][fov][:, baselinefirstframe:baselinelastframe], axis=1)
            popevents_fov[day][animal][fov] = popevents_fov[day][animal][fov] - baseline[:, None]
for day in sorted(days):
    popevents_day[day] = popevents_day[day][1:,:]
    baseline = np.mean(popevents_day[day][:, baselinefirstframe:baselinelastframe], axis=1)
    popevents_day[day] = popevents_day[day] - baseline[:, None]
popevents_all = np.vstack([popevents_day[day] for day in sorted(days)])

0 EarlyAcq


AttributeError: module 'numpy' has no attribute 'NAN'

In [ ]:
aucpvals_day = {}
aucpvals_fov = {}
if save_auc_array == 'yes':
    for basedir, day, animal, fov in iterate_dirs(basedir, sorted(days)):
        tempfiles = next(os.walk(os.path.join(basedir, day, animal, fov)))[2]
        if 'alignedevents_%s_%s.npy'%(eventofinterest, separation_requirement) in tempfiles: 
            temp2 = np.load(os.path.join(basedir, day, animal, fov, 'alignedevents_%s_%s.npy'%(eventofinterest, separation_requirement)))
            if np.isfinite(np.nanmean(temp2)):
                aucpvals_fov_temp = np.nan*np.ones((temp2.shape[0], 2))
                baseline = np.mean(temp2[:,baselinefirstframe:baselinelastframe, :], axis=1)
                alignedevents = temp2 - baseline[:,None,:]
                newbaseline = np.mean(alignedevents[:,baselinefirstframe:baselinelastframe, :], axis=1)
                event = np.mean(alignedevents[:,aucfirstframe:auclastframe, :], axis=1)
                for neuron in range(np.shape(temp2)[0]):
                    auc, pval = calculate_auROC(event[neuron,:], newbaseline[neuron,:])
                    aucpvals_fov_temp[neuron,0] = auc
                    aucpvals_fov_temp[neuron,1] = pval
for basedir, day, animal, fov in iterate_dirs(basedir, sorted(days)):
    if day not in aucpvals_fov:
        aucpvals_day[day] = np.nan*np.ones((1, 2))
        aucpvals_fov[day] = {}
        print(day)
    tempfiles = next(os.walk(os.path.join(basedir, day, animal, fov)))[2]
    if 'alignedevents_%s_%s.npy'%(eventofinterest, separation_requirement) in tempfiles: 
        if animal not in aucpvals_fov[day]:
            aucpvals_fov[day][animal] = {}
        aucpvals_fov[day][animal][fov] = np.load(os.path.join(basedir, day, animal, fov, 'aucpvals_%s_%s.npy'%(eventofinterest, separation_requirement)))
        aucpvals_day[day] = np.vstack((aucpvals_day[day], aucpvals_fov[day][animal][fov]))
for day in sorted(days):
    aucpvals_day[day] = aucpvals_day[day][1:, :]  # Remove the first row of NaNs
aucpvals_all = np.vstack([aucpvals_day[day] for day in sorted(days)])

In [ ]:
neuronstoplot = 'all'
plot_zscored_popevents = 'yes'
if numdays > 1:
    fig, axs = plt.subplots(2, numdays, figsize=(15, 8))
else: 
    fig, axs = plt.subplots(2, 2, figsize=(5, 8))
sns.set_style('white')
cmax = 4
cmin = -cmax
ymax = 4
ymin = -ymax
for d, day in enumerate(sorted(days)):
    if bh_correction == 'yes':
        aucpvals_day[day][:,1] = Benjamini_Hochberg_correction(aucpvals_day[day][:,1])
    if neuronstoplot == 'all':
        temp = np.array(popevents_day[day])
    elif neuronstoplot == 'significant':
        temp = popevents_day[day][np.where(aucpvals_day[day][:,1] <= alphalevel)[0],:] 
    elif neuronstoplot == 'notsignificant':
        temp = popevents_day[day][np.where(aucpvals_day[day][:,1] > alphalevel)[0],:]  
    if plot_zscored_popevents == 'yes':
        temp_std = []
        for neuron in range(temp.shape[0]):
            neuron_std = np.nanstd(temp[neuron, baselinefirstframe:baselinelastframe])
            temp_std = np.hstack((temp_std, neuron_std))
            temp[neuron, :] = temp[neuron, :]/neuron_std
        if d == 0:
            print('data z-scored')
    tempresponse = np.nanmean(temp, axis=1)
    sortresponse = np.argsort(tempresponse)[::-1]
    numneurons_temp = len(sortresponse)
    ax = axs[0, d]
    im = ax.imshow(temp[sortresponse], cmap=plt.get_cmap('PRGn_r'), vmin=cmin, vmax=cmax, aspect='auto')
    ax.grid(False)
    ax.set_ylabel('%s neurons'%numneurons_temp)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.plot([pre_window_size, pre_window_size], 
            [0, numneurons_temp], '--k', linewidth=1.5)
    ax.plot([infusionframe, infusionframe],
            [0, numneurons_temp], '--k', linewidth=1.5)
    ax = axs[1, d]
    ax.set_xticks([])
    ax.plot(np.mean(temp, axis = 0))
    ax.plot([pre_window_size, pre_window_size],
             [-0.5,2.5],'--k', linewidth=1.5)
    ax.plot([0, window_size],
            [0,0], '--k', linewidth=0.5)
fig.tight_layout()
plt.show()